In [123]:
import pandas as pd

df_profiles = pd.read_pickle("profiles_with_embeddings.pkl")
dfpr = pd.read_pickle("dfpr_with_embeddings.pkl")

In [132]:
df_profiles.info()

<class 'pandas.DataFrame'>
RangeIndex: 284 entries, 0 to 283
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   folder_number       284 non-null    int64 
 1   profile_dir         284 non-null    str   
 2   analysis_path       284 non-null    str   
 3   profile_id          284 non-null    str   
 4   username            284 non-null    str   
 5   full_name           284 non-null    str   
 6   followers           284 non-null    int64 
 7   blogger_gender      284 non-null    str   
 8   domain              284 non-null    str   
 9   keywords            284 non-null    str   
 10  entities            284 non-null    str   
 11  brands              284 non-null    str   
 12  video_format        284 non-null    str   
 13  visual_style        284 non-null    str   
 14  audio_and_delivery  284 non-null    str   
 15  vibe                284 non-null    str   
 16  rendered_text       284 non-null    s

In [133]:
dfpr.info()

<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 252
Data columns (total 6 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Описание рекламной кампании (Бриф для ИИ)  43 non-null     str   
 1   Продукт / Услуга                           43 non-null     str   
 2   Формат видео                               43 non-null     str   
 3   Tone of Voice (ToV)                        43 non-null     str   
 4   campaign_text                              43 non-null     str   
 5   embedding                                  43 non-null     object
dtypes: object(1), str(5)
memory usage: 89.9+ KB


In [124]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, ndcg_score
from sklearn.base import BaseEstimator, RegressorMixin

In [125]:
MARKUP_PATH = "Final разметка AI-Score.xlsx"  # или /mnt/data/Final разметка AI-Score(2).xlsx

df_markup = pd.read_excel(MARKUP_PATH)

df_markup = df_markup.rename(columns={
    "Оценка": "ai_score"
})

df_markup["campaign_original_index"] = df_markup["campaign_original_index"].astype(int)
df_markup["blogger_folder_number"] = df_markup["blogger_folder_number"].astype(int)
df_markup["ai_score"] = df_markup["ai_score"].astype(float)

# Проверка: должно быть 7 строк на кампанию
rows_per_campaign = df_markup.groupby("campaign_original_index").size()
assert rows_per_campaign.eq(7).all(), rows_per_campaign[~rows_per_campaign.eq(7)]

df_markup.head()

,campaign_original_index,campaign_name,campaign_format,campaign_tov,campaign_brief,match_type,rank,score,blogger_folder_number,blogger_profile_id,...,blogger_domain,blogger_keywords,blogger_entities,blogger_visual_style,blogger_brands,blogger_video_format,blogger_audio_and_delivery,blogger_vibe,blogger_instagram_url,ai_score
0,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,1,0.658765,16,62463700056,...,"уход за волосами, бьюти-лайфстайл, прически, у...","плетение кос, легкая прическа, пучок из волос,...","СПБ, Эмили","крупный план лица, средний план сзади, комнатн...",ARAVIA Laboratories,"Инструкции / Лайфхаки, GRWM (Get Ready With Me...","динамичная фоновая музыка, трендовые звуковые ...","уютная атмосфера, дружелюбный настрой, эстетич...",https://www.instagram.com/alina.chichikalova/,53.0
1,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,2,0.657407,137,58146336414,...,"бьюти, уход за кожей, макияж, лайфстайл, стиль...","сыворотка для лица, крем для лица SPF, солевой...","СПБ, Питер, Севкабель, Света, Влад","крупный план, селфи-видео, съемка в зеркало, м...","Skinfood, bonyasbox, Naedine Studio, Elastica,...","GRWM, Инструкции / Лайфхаки, день из жизни, бь...","закадровый голос, спокойный тембр, мягкая пода...","эстетичный, расслабленный, романтичный, уютный...",https://www.instagram.com/sveta.subbo?igsh=MWk...,51.0
2,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,3,0.647581,251,5772898877,...,"мода, косметика, бьюти, уход за кожей, моделин...","коллагеновые патчи под глаза, крем для кожи во...",Москва,"крупный план, теплое домашнее освещение, демон...","Abib, Yasoma, Lunifera, VT Cosmetics","GRWM (Get Ready With Me), Распаковка, Обзор, у...","приятный женский голос, динамичная фоновая муз...","эстетичный, вдохновляющий, бьюти-атмосфера, за...",https://www.instagram.com/plkarter/,51.0
3,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,4,0.641223,39,4013884722,...,"бьюти-блог, уход за кожей, UGC-контент, лайфст...","сыворотка для лица, спф-крем для лица, СС-крем...","Saint Petersburg, СПб, Наташа","крупный план, дневное освещение, эстетичные ка...","Tiam, Luxvisage, Essence, Beauty Bomb, Eva Mos...","GRWM, туториал по макияжу, лайфстайл влог, обз...","закадровый голос, спокойный тембр, фоновая муз...","уютный, легкий, эстетичный, расслабленный, лет...",https://www.instagram.com/katena.astahowa/reels/,52.0
4,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,5,0.640058,202,415772084,...,"лайфстайл, бьюти-индустрия, создание UGC конте...","утренняя рутина, тканевая маска для лица, увла...","Москва, Аня","эстетичный видеоряд, мягкий дневной свет, тепл...","Celimax, Boostlife, Reshape, Atmosphere, Skims...","GRWM (Get Ready With Me), повседневный влог, д...","приятный закадровый голос, спокойная манера ре...","уютная атмосфера, эстетичный минимализм, вдохн...",https://www.instagram.com/annetdnk?igsh=MXVhan...,68.0


In [126]:
def prepare_campaigns(dfpr: pd.DataFrame) -> pd.DataFrame:
    campaigns = dfpr.copy()

    # В твоем dfpr индексы похожи на 0, 6, 12, ..., 252.
    # Поэтому сохраняем исходный индекс как campaign_original_index.
    if "campaign_original_index" not in campaigns.columns:
        campaigns["campaign_original_index"] = campaigns.index

    campaigns = campaigns.rename(columns={"embedding": "campaign_embedding"})
    campaigns["campaign_original_index"] = campaigns["campaign_original_index"].astype(int)

    return campaigns


def prepare_profiles(df_profiles: pd.DataFrame) -> pd.DataFrame:
    profiles = df_profiles.copy()

    profiles = profiles.rename(columns={
        "embedding": "profile_embedding",
        "folder_number": "blogger_folder_number"
    })

    profiles["blogger_folder_number"] = profiles["blogger_folder_number"].astype(int)

    return profiles


campaigns = prepare_campaigns(dfpr)
profiles = prepare_profiles(df_profiles)

In [127]:
def to_vec(x) -> np.ndarray:
    """
    Поддерживает np.array, list, tuple и строку вида '[0.1 0.2 ...]'.
    """
    if isinstance(x, np.ndarray):
        return x.astype(float)

    if isinstance(x, (list, tuple)):
        return np.array(x, dtype=float)

    if isinstance(x, str):
        s = x.strip()
        s = s.replace("\n", " ")
        s = s.strip("[]")
        return np.fromstring(s, sep=" ")

    raise TypeError(f"Unsupported embedding type: {type(x)}")


def embedding_features(campaign_emb, profile_emb) -> dict:
    c = to_vec(campaign_emb)
    p = to_vec(profile_emb)

    assert c.shape == p.shape, f"Different embedding shapes: {c.shape} vs {p.shape}"

    c_norm = np.linalg.norm(c)
    p_norm = np.linalg.norm(p)

    dot = float(np.dot(c, p))
    cosine = dot / (c_norm * p_norm + 1e-12)

    diff = c - p
    abs_diff = np.abs(diff)
    prod = c * p

    return {
        "emb_cosine": cosine,
        "emb_cosine_distance": 1 - cosine,
        "emb_dot": dot,
        "emb_euclidean": float(np.linalg.norm(diff)),
        "emb_manhattan": float(abs_diff.sum()),
        "emb_chebyshev": float(abs_diff.max()),
        "emb_c_norm": float(c_norm),
        "emb_p_norm": float(p_norm),
        "emb_norm_ratio": float(c_norm / (p_norm + 1e-12)),
        "emb_absdiff_mean": float(abs_diff.mean()),
        "emb_absdiff_std": float(abs_diff.std()),
        "emb_absdiff_max": float(abs_diff.max()),
        "emb_prod_mean": float(prod.mean()),
        "emb_prod_std": float(prod.std()),
        "emb_prod_sum": float(prod.sum()),
    }

In [128]:
PROFILE_JSON_ROOT = Path(".")  
# пример:
# PROFILE_JSON_ROOT = Path("data/instagram_profiles")

In [129]:
def safe_num(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def stat_features(values, prefix):
    values = np.array([safe_num(v) for v in values], dtype=float)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {
            f"{prefix}_sum": np.nan,
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_min": np.nan,
            f"{prefix}_std": np.nan,
        }

    return {
        f"{prefix}_sum": float(values.sum()),
        f"{prefix}_mean": float(values.mean()),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_max": float(values.max()),
        f"{prefix}_min": float(values.min()),
        f"{prefix}_std": float(values.std()),
    }


def find_profile_json(folder_number, root: Path) -> Path | None:
    folder = root / str(int(folder_number))

    direct = folder / "profile.json"
    if direct.exists():
        return direct

    # запасной вариант, если profile.json лежит глубже
    matches = list(folder.glob("**/profile.json"))
    if matches:
        return matches[0]

    return None


def profile_json_features(folder_number, root: Path) -> dict:
    path = find_profile_json(folder_number, root)

    base = {
        "blogger_folder_number": int(folder_number),
        "json_found": 0,

        # новые поля
        "profile_url": np.nan,
        "link": np.nan,
        "err": np.nan,
        "ERR": np.nan,
        "avg_views": np.nan,
        "avg_likes": np.nan,
    }

    if path is None:
        base["err"] = "profile_json_not_found"
        base["ERR"] = "profile_json_not_found"
        return base

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        base["err"] = str(e)
        base["ERR"] = str(e)
        return base

    videos = data.get("videos") or []

    likes = [v.get("likes") for v in videos]
    comments = [v.get("comments") for v in videos]
    views = [v.get("views") for v in videos]

    captions = [v.get("caption") or "" for v in videos]

    caption_lengths = [len(c) for c in captions]
    hashtags_count = [len(re.findall(r"#\w+", c)) for c in captions]
    mentions_count = [len(re.findall(r"@\w+", c)) for c in captions]

    likes_arr = np.array([safe_num(x) for x in likes], dtype=float)
    comments_arr = np.array([safe_num(x) for x in comments], dtype=float)
    views_arr = np.array([safe_num(x) for x in views], dtype=float)

    engagement = (likes_arr + comments_arr) / np.maximum(views_arr, 1)

    timestamps = pd.to_datetime(
        [v.get("timestamp") for v in videos],
        errors="coerce",
        utc=True
    ).dropna()

    if len(timestamps) > 0:
        last_post_ts = timestamps.max()
        first_post_ts = timestamps.min()
        posting_span_days = max((last_post_ts - first_post_ts).days, 0)
        days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
    else:
        posting_span_days = np.nan
        days_since_last_post = np.nan

    followers = safe_num(data.get("followers"))
    following = safe_num(data.get("following"))
    posts_count = safe_num(data.get("posts_count"))

    downloaded_flags = [
        1 if v.get("downloaded") is True else 0
        for v in videos
    ]

    profile_url = data.get("profile_url")

    out = {
        **base,

        "json_found": 1,

        # новые поля
        "profile_url": profile_url,
        "link": profile_url,

        # если в json есть такие поля — заберём
        "err": data.get("err", data.get("ERR", np.nan)),
        "ERR": data.get("ERR", data.get("err", np.nan)),

        "avg_views": float(np.nanmean(views_arr)) if len(views_arr) else np.nan,
        "avg_likes": float(np.nanmean(likes_arr)) if len(likes_arr) else np.nan,

        "json_followers": followers,
        "json_following": following,
        "json_posts_count": posts_count,

        "json_verified": int(bool(data.get("verified"))),
        "json_private": int(bool(data.get("private"))),
        "json_business_account": int(bool(data.get("business_account"))),

        "json_videos_count": len(videos),
        "json_following_to_followers": following / (followers + 1),
        "json_posts_to_followers": posts_count / (followers + 1),

        "json_days_since_last_post": days_since_last_post,
        "json_posting_span_days": posting_span_days,
        "json_posts_per_30d_in_sample": (
            len(videos) / (posting_span_days / 30 + 1)
            if not pd.isna(posting_span_days)
            else np.nan
        ),

        "json_downloaded_share": np.mean(downloaded_flags) if len(downloaded_flags) else np.nan,

        "json_engagement_mean": float(np.nanmean(engagement)) if len(engagement) else np.nan,
        "json_engagement_median": float(np.nanmedian(engagement)) if len(engagement) else np.nan,
    }

    out.update(stat_features(likes, "json_likes"))
    out.update(stat_features(comments, "json_comments"))
    out.update(stat_features(views, "json_views"))
    out.update(stat_features(caption_lengths, "json_caption_len"))
    out.update(stat_features(hashtags_count, "json_hashtags"))
    out.update(stat_features(mentions_count, "json_mentions"))

    return out


profile_json_df = pd.DataFrame([
    profile_json_features(folder, PROFILE_JSON_ROOT)
    for folder in profiles["blogger_folder_number"].dropna().unique()
])

profile_json_df.head()

C:\Users\user\AppData\Local\Temp\ipykernel_34452\1367357950.py:106: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_34452\1367357950.py:106: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_34452\1367357950.py:106: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Timestamp.utcnow() - last_post_ts).days
C:\Users\user\AppData\Local\Temp\ipykernel_34452\1367357950.py:106: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  days_since_last_post = (pd.Time

,blogger_folder_number,json_found,profile_url,link,err,ERR,avg_views,avg_likes,json_followers,json_following,...,json_hashtags_median,json_hashtags_max,json_hashtags_min,json_hashtags_std,json_mentions_sum,json_mentions_mean,json_mentions_median,json_mentions_max,json_mentions_min,json_mentions_std
0,1,1,https://www.instagram.com/gvardis.m,https://www.instagram.com/gvardis.m,NaN,NaN,2066.111111,74.444444,3780.0,391.0,...,1.0,5.0,0.0,2.078699,2.0,0.222222,0.0,1.0,0.0,0.415740
1,4,1,https://www.instagram.com/as_crimeaa,https://www.instagram.com/as_crimeaa,NaN,NaN,311165.833333,32434.083333,10228.0,184.0,...,0.0,4.0,0.0,1.462494,3.0,0.250000,0.0,1.0,0.0,0.433013
2,5,1,https://www.instagram.com/pon.alexa,https://www.instagram.com/pon.alexa,NaN,NaN,84364.909091,4811.545455,13875.0,224.0,...,0.0,0.0,0.0,0.000000,1.0,0.090909,0.0,1.0,0.0,0.287480
3,7,1,https://www.instagram.com/kate_mom2,https://www.instagram.com/kate_mom2,NaN,NaN,491.000000,23.583333,1793.0,211.0,...,0.0,5.0,0.0,1.907587,1.0,0.083333,0.0,1.0,0.0,0.276385
4,9,1,https://www.instagram.com/aammmmm79,https://www.instagram.com/aammmmm79,NaN,NaN,2355.400000,191.000000,1622.0,891.0,...,0.0,8.0,0.0,3.487119,5.0,1.000000,1.0,3.0,0.0,1.095445


In [130]:
pairs = (
    df_markup
    .merge(
        campaigns[["campaign_original_index", "campaign_embedding"]],
        on="campaign_original_index",
        how="left"
    )
    .merge(
        profiles[["blogger_folder_number", "profile_embedding"]],
        on="blogger_folder_number",
        how="left"
    )
    .merge(
        profile_json_df,
        on="blogger_folder_number",
        how="left"
    )
)

assert pairs["campaign_embedding"].notna().all(), "Есть кампании без campaign_embedding"
assert pairs["profile_embedding"].notna().all(), "Есть блогеры без profile_embedding"

pairs["target_prob"] = pairs["ai_score"] / 100.0

pairs.shape

(301, 86)

In [131]:
pairs.info()

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 86 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   campaign_original_index       301 non-null    int64  
 1   campaign_name                 301 non-null    str    
 2   campaign_format               301 non-null    str    
 3   campaign_tov                  301 non-null    str    
 4   campaign_brief                301 non-null    str    
 5   match_type                    301 non-null    str    
 6   rank                          301 non-null    int64  
 7   score                         301 non-null    float64
 8   blogger_folder_number         301 non-null    int64  
 9   blogger_profile_id            301 non-null    int64  
 10  blogger_username              301 non-null    str    
 11  blogger_full_name             291 non-null    str    
 12  blogger_followers             301 non-null    int64  
 13  blogger_profile_

In [94]:
emb_features = pd.DataFrame([
    embedding_features(c_emb, p_emb)
    for c_emb, p_emb in zip(pairs["campaign_embedding"], pairs["profile_embedding"])
])

pairs = pd.concat([pairs.reset_index(drop=True), emb_features], axis=1)

In [95]:
pairs["retrieval_score"] = pairs["score"].astype(float)
pairs["retrieval_rank"] = pairs["rank"].astype(float)
pairs["blogger_followers_log"] = np.log1p(pairs["blogger_followers"].astype(float))

# Количества текстовых сущностей из уже готового анализа профиля
for col in [
    "blogger_domain",
    "blogger_keywords",
    "blogger_entities",
    "blogger_visual_style",
    "blogger_brands",
    "blogger_video_format",
    "blogger_audio_and_delivery",
    "blogger_vibe",
]:
    pairs[f"{col}_items_count"] = (
        pairs[col]
        .fillna("")
        .astype(str)
        .apply(lambda s: len([x for x in s.split(",") if x.strip()]))
    )

# Пол блогера — one-hot
gender_dummies = pd.get_dummies(
    pairs["blogger_gender"].fillna("unknown"),
    prefix="gender",
    dtype=int
)

pairs = pd.concat([pairs, gender_dummies], axis=1)

In [96]:
def choose_validation_campaigns(
    df,
    campaign_col="campaign_original_index",
    target_col="ai_score",
    n_val_campaigns=7,
    must_val=252,
    must_train=246,
):
    all_campaigns = sorted(df[campaign_col].unique())

    assert must_val in all_campaigns, f"{must_val} нет в кампаниях"
    assert must_train in all_campaigns, f"{must_train} нет в кампаниях"
    assert must_val != must_train

    # Выбираем остальные 6 кампаний так, чтобы они были распределены
    # по среднему ai_score: от низких к высоким.
    campaign_mean = (
        df.groupby(campaign_col)[target_col]
        .mean()
        .drop(index=[must_val, must_train], errors="ignore")
        .sort_values()
    )

    candidates = campaign_mean.index.to_list()

    need = n_val_campaigns - 1
    positions = np.linspace(0, len(candidates) - 1, need).round().astype(int)

    picked = []
    used = set()

    for pos in positions:
        # защита от дублей после round()
        for shift in range(len(candidates)):
            for idx in [pos - shift, pos + shift]:
                if 0 <= idx < len(candidates):
                    c = candidates[idx]
                    if c not in used:
                        picked.append(c)
                        used.add(c)
                        break
            if len(picked) == len(used):
                break

    val_campaigns = sorted([must_val] + picked[:need])
    return val_campaigns


VAL_CAMPAIGNS = choose_validation_campaigns(
    pairs,
    n_val_campaigns=7,
    must_val=252,
    must_train=246,
)

TRAIN_MUST_CAMPAIGN = 246

val_mask = pairs["campaign_original_index"].isin(VAL_CAMPAIGNS)

train_pairs = pairs.loc[~val_mask].copy()
val_pairs = pairs.loc[val_mask].copy()

assert 252 in set(val_pairs["campaign_original_index"]), "252 не попала в validation"
assert 246 in set(train_pairs["campaign_original_index"]), "246 не попала в train"
assert 246 not in set(val_pairs["campaign_original_index"]), "246 ошибочно попала в validation"
assert len(VAL_CAMPAIGNS) == 7, VAL_CAMPAIGNS
assert len(val_pairs) == 49, len(val_pairs)

print("Validation campaigns:", VAL_CAMPAIGNS)
print("Train rows:", len(train_pairs))
print("Validation rows:", len(val_pairs))

Validation campaigns: [66, 78, 84, 132, 168, 240, 252]
Train rows: 252
Validation rows: 49


In [97]:
class SoftLabelLogisticRegression(BaseEstimator, RegressorMixin):
    def __init__(self, C=1.0, fit_intercept=True, max_iter=2000, tol=1e-7):
        self.C = C
        self.fit_intercept = fit_intercept
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        y = np.clip(y, 0.0, 1.0)

        n, d = X.shape

        if self.fit_intercept:
            X_design = np.column_stack([np.ones(n), X])
        else:
            X_design = X

        alpha = 1.0 / max(self.C, 1e-12)

        def objective(beta):
            z = X_design @ beta
            p = expit(z)

            eps = 1e-12
            loss = -np.mean(
                y * np.log(p + eps) +
                (1 - y) * np.log(1 - p + eps)
            )

            if self.fit_intercept:
                reg_beta = beta[1:]
            else:
                reg_beta = beta

            loss += 0.5 * alpha * np.sum(reg_beta ** 2) / n

            grad = X_design.T @ (p - y) / n

            if self.fit_intercept:
                grad[1:] += alpha * beta[1:] / n
            else:
                grad += alpha * beta / n

            return loss, grad

        beta0 = np.zeros(X_design.shape[1], dtype=float)

        res = minimize(
            fun=lambda b: objective(b)[0],
            x0=beta0,
            jac=lambda b: objective(b)[1],
            method="L-BFGS-B",
            options={
                "maxiter": self.max_iter,
                "ftol": self.tol,
            },
        )

        self.coef_full_ = res.x
        self.optimization_result_ = res

        if self.fit_intercept:
            self.intercept_ = np.array([res.x[0]])
            self.coef_ = res.x[1:].reshape(1, -1)
        else:
            self.intercept_ = np.array([0.0])
            self.coef_ = res.x.reshape(1, -1)

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        z = X @ self.coef_.ravel() + self.intercept_[0]
        p = expit(z)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return self.predict_proba(X)[:, 1]

In [98]:
feature_cols = [
    # embedding distances
    "emb_cosine",
    "emb_cosine_distance",
    "emb_dot",
    "emb_euclidean",
    "emb_manhattan",
    "emb_chebyshev",
    "emb_c_norm",
    "emb_p_norm",
    "emb_norm_ratio",
    "emb_absdiff_mean",
    "emb_absdiff_std",
    "emb_absdiff_max",
    "emb_prod_mean",
    "emb_prod_std",
    "emb_prod_sum",

    # retrieval features
    "retrieval_score",
    "retrieval_rank",

    # basic profile features
    "blogger_followers_log",

    # json profile features
    "json_found",
    "json_followers",
    "json_following",
    "json_posts_count",
    "json_verified",
    "json_private",
    "json_business_account",
    "json_videos_count",
    "json_following_to_followers",
    "json_posts_to_followers",
    "json_days_since_last_post",
    "json_posting_span_days",
    "json_posts_per_30d_in_sample",
    "json_downloaded_share",
    "json_engagement_mean",
    "json_engagement_median",

    "json_likes_sum",
    "json_likes_mean",
    "json_likes_median",
    "json_likes_max",
    "json_likes_min",
    "json_likes_std",

    "json_comments_sum",
    "json_comments_mean",
    "json_comments_median",
    "json_comments_max",
    "json_comments_min",
    "json_comments_std",

    "json_views_sum",
    "json_views_mean",
    "json_views_median",
    "json_views_max",
    "json_views_min",
    "json_views_std",

    "json_caption_len_mean",
    "json_caption_len_median",
    "json_caption_len_max",
    "json_hashtags_mean",
    "json_mentions_mean",
]

# Добавляем one-hot gender и count-признаки
feature_cols += [c for c in pairs.columns if c.startswith("gender_")]
feature_cols += [c for c in pairs.columns if c.endswith("_items_count")]

# Оставляем только существующие колонки
feature_cols = [c for c in feature_cols if c in pairs.columns]

# Убираем признаки, которые полностью пустые в train
feature_cols = [
    c for c in feature_cols
    if not train_pairs[c].isna().all()
]

X_train = train_pairs[feature_cols]
y_train = train_pairs["target_prob"]

X_val = val_pairs[feature_cols]
y_val = val_pairs["target_prob"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", SoftLabelLogisticRegression(C=1.0, max_iter=3000)),
])

model.fit(X_train, y_train)

val_pred_prob = model.predict(X_val)
val_pred_score = np.clip(val_pred_prob * 100, 0, 100)

In [99]:
val_eval = val_pairs[[
    "campaign_original_index",
    "campaign_name",
    "blogger_folder_number",
    "blogger_username",
    "blogger_full_name",
    "ai_score",
    "score",
    "rank",
]].copy()

val_eval["pred_prob"] = val_pred_prob
val_eval["pred_ai_score"] = val_pred_score
val_eval["abs_error"] = np.abs(val_eval["ai_score"] - val_eval["pred_ai_score"])

val_eval = val_eval.sort_values(
    ["campaign_original_index", "pred_ai_score"],
    ascending=[True, False]
)

val_eval.head(20)

,campaign_original_index,campaign_name,blogger_folder_number,blogger_username,blogger_full_name,ai_score,score,rank,pred_prob,pred_ai_score,abs_error
81,66,Жидкий тинт для губ и щек,136,milkinadi,Диана Милкина / контент креатор • стиль • лайф...,47.0,0.669001,5,0.683243,68.324335,21.324335
77,66,Жидкий тинт для губ и щек,90,za_makeupom,VICTORIA шопоголик🛍️,59.0,0.681653,1,0.611705,61.170470,2.170470
78,66,Жидкий тинт для губ и щек,227,marrrielka,Мария | beauty influencer | UGC creator | СПБ ...,55.0,0.676098,2,0.561361,56.136098,1.136098
80,66,Жидкий тинт для губ и щек,93,evochie2003,"Ева, 22 года, дева, королева",65.0,0.669037,4,0.536574,53.657418,11.342582
79,66,Жидкий тинт для губ и щек,65,eegorvvva,леруа | 📍novosibirsk,43.0,0.671029,3,0.406192,40.619166,2.380834
83,66,Жидкий тинт для губ и щек,180,ooshroevaa,"Натали, блогер из Краснодара",16.0,0.557396,2,0.188275,18.827484,2.827484
82,66,Жидкий тинт для губ и щек,102,luukovka,обзоры wb| influencer | UGC - креатор,5.0,0.551487,1,0.104611,10.461138,5.461138
92,78,"Базовый трикотаж (футболки, боди)",242,mokka.ira,"Ира | жизнь как из Пинтерест, но настоящая",66.0,0.659349,2,0.631331,63.133099,2.866901
93,78,"Базовый трикотаж (футболки, боди)",186,yannadmi,"Яна | обзоры и распаковки & Wb, OZON, ЯМ",63.0,0.658466,3,0.545672,54.567167,8.432833
91,78,"Базовый трикотаж (футболки, боди)",26,lyyuovy,Любовь - beauty & lifestyle|ugc|Воронеж,65.0,0.663091,1,0.521989,52.198892,12.801108


In [100]:
def campaign_ndcg(g):
    return ndcg_score(
        [g["ai_score"].values],
        [g["pred_ai_score"].values]
    )

ndcg_by_campaign = (
    val_eval
    .groupby("campaign_original_index")
    .apply(campaign_ndcg)
    .rename("ndcg")
    .reset_index()
)

print("Mean NDCG:", ndcg_by_campaign["ndcg"].mean())
ndcg_by_campaign

Mean NDCG: 0.8919918736871238


,campaign_original_index,ndcg
0,66,0.941093
1,78,0.998392
2,84,0.985192
3,132,0.983712
4,168,0.984155
5,240,0.703181
6,252,0.648218


In [101]:
train_pred_prob = model.predict(X_train)
train_pred_score = np.clip(train_pred_prob * 100, 0, 100)

In [102]:
train_eval = train_pairs[[
    "campaign_original_index",
    "campaign_name",
    "blogger_folder_number",
    "blogger_username",
    "blogger_full_name",
    "ai_score",
    "score",
    "rank",
]].copy()

train_eval["pred_prob"] = train_pred_prob
train_eval["pred_ai_score"] = train_pred_score
train_eval["abs_error"] = np.abs(train_eval["ai_score"] - train_eval["pred_ai_score"])

train_eval = train_eval.sort_values(
    ["campaign_original_index", "pred_ai_score"],
    ascending=[True, False]
)

In [103]:
from sklearn.metrics import ndcg_score

def campaign_ndcg(g):
    return ndcg_score(
        [g["ai_score"].values],
        [g["pred_ai_score"].values]
    )

def calc_ndcg_by_campaign(eval_df, name):
    ndcg_by_campaign = (
        eval_df
        .groupby("campaign_original_index")
        .apply(campaign_ndcg)
        .rename("ndcg")
        .reset_index()
    )

    mean_ndcg = ndcg_by_campaign["ndcg"].mean()

    print(f"{name} Mean NDCG: {mean_ndcg:.4f}")

    return ndcg_by_campaign, mean_ndcg


train_ndcg_by_campaign, train_mean_ndcg = calc_ndcg_by_campaign(train_eval, "Train")
val_ndcg_by_campaign, val_mean_ndcg = calc_ndcg_by_campaign(val_eval, "Validation")

Train Mean NDCG: 0.9694
Validation Mean NDCG: 0.8920


In [104]:
def campaign_ndcg(g):
    return ndcg_score(
        [g["ai_score"].values],
        [g["pred_ai_score"].values]
    )

ndcg_by_campaign = (
    val_eval
    .groupby("campaign_original_index")
    .apply(campaign_ndcg)
    .rename("ndcg")
    .reset_index()
)

print("Mean NDCG:", ndcg_by_campaign["ndcg"].mean())
ndcg_by_campaign

Mean NDCG: 0.8919918736871238


,campaign_original_index,ndcg
0,66,0.941093
1,78,0.998392
2,84,0.985192
3,132,0.983712
4,168,0.984155
5,240,0.703181
6,252,0.648218


In [105]:
val_eval.to_excel("validation_predictions_soft_logreg.xlsx", index=False)

pairs.to_pickle("pairs_with_features_for_logreg.pkl")
train_pairs.to_pickle("train_pairs.pkl")
val_pairs.to_pickle("val_pairs.pkl")

In [106]:
coef = pd.Series(
    model.named_steps["logreg"].coef_.ravel(),
    index=feature_cols
)

coef_abs = (
    coef
    .to_frame("coef")
    .assign(abs_coef=lambda x: x["coef"].abs())
    .sort_values("abs_coef", ascending=False)
)

coef_abs.head(30)

,coef,abs_coef
json_views_min,-0.617626,0.617626
blogger_entities_items_count,-0.313841,0.313841
gender_мужской,-0.285534,0.285534
gender_женский,0.243948,0.243948
blogger_brands_items_count,0.222094,0.222094
json_likes_min,0.211414,0.211414
blogger_keywords_items_count,-0.197817,0.197817
json_likes_median,0.186229,0.186229
blogger_visual_style_items_count,0.177141,0.177141
blogger_video_format_items_count,-0.173684,0.173684


In [107]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    ndcg_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

In [108]:
X_train_cb = train_pairs[feature_cols].copy()
X_val_cb = val_pairs[feature_cols].copy()

# На всякий случай приводим все признаки к числам
for col in feature_cols:
    X_train_cb[col] = pd.to_numeric(X_train_cb[col], errors="coerce")
    X_val_cb[col] = pd.to_numeric(X_val_cb[col], errors="coerce")

y_train_prob = train_pairs["target_prob"].astype(float)
y_val_prob = val_pairs["target_prob"].astype(float)

train_pool = Pool(
    data=X_train_cb,
    label=y_train_prob,
    feature_names=feature_cols,
)

val_pool = Pool(
    data=X_val_cb,
    label=y_val_prob,
    feature_names=feature_cols,
)

In [116]:
X_train_cb

,emb_cosine,emb_cosine_distance,emb_dot,emb_euclidean,emb_manhattan,emb_chebyshev,emb_c_norm,emb_p_norm,emb_norm_ratio,emb_absdiff_mean,...,gender_мужской,gender_универсальный,blogger_domain_items_count,blogger_keywords_items_count,blogger_entities_items_count,blogger_visual_style_items_count,blogger_brands_items_count,blogger_video_format_items_count,blogger_audio_and_delivery_items_count,blogger_vibe_items_count
0,0.658765,0.341235,0.658765,0.826117,21.160168,0.086501,1.0,1.0,1.0,0.020664,...,0,0,9,12,2,8,1,7,7,7
1,0.657407,0.342593,0.657407,0.827760,21.273925,0.092491,1.0,1.0,1.0,0.020775,...,0,0,12,25,5,8,9,6,8,8
2,0.647581,0.352419,0.647581,0.839546,21.429672,0.086186,1.0,1.0,1.0,0.020927,...,0,0,10,14,1,7,4,7,7,7
3,0.641223,0.358777,0.641223,0.847086,21.779458,0.093506,1.0,1.0,1.0,0.021269,...,0,0,8,13,3,7,7,5,6,8
4,0.640058,0.359942,0.640058,0.848460,21.466611,0.098502,1.0,1.0,1.0,0.020963,...,0,0,8,18,2,6,9,5,7,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,0.584474,0.415526,0.584474,0.911620,22.985701,0.107106,1.0,1.0,1.0,0.022447,...,0,0,10,16,2,11,8,8,10,11
290,0.578089,0.421911,0.578089,0.918598,23.508958,0.095315,1.0,1.0,1.0,0.022958,...,0,0,11,14,5,6,2,7,7,9
291,0.574890,0.425110,0.574890,0.922074,23.383821,0.092731,1.0,1.0,1.0,0.022836,...,0,0,12,14,3,7,5,6,7,7
292,0.500905,0.499095,0.500905,0.999095,25.461746,0.115444,1.0,1.0,1.0,0.024865,...,1,0,12,13,15,10,12,5,10,8


In [109]:
cat_model = CatBoostClassifier(
    loss_function="CrossEntropy",
    eval_metric="CrossEntropy",

    iterations=2000,
    learning_rate=0.03,
    depth=3,

    l2_leaf_reg=10,
    random_strength=1.0,
    bagging_temperature=1.0,

    random_seed=42,
    verbose=100,
    allow_writing_files=False,
)

cat_model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True,
    early_stopping_rounds=100,
)

0:	learn: 0.6871197	test: 0.6855491	best: 0.6855491 (0)	total: 5.64ms	remaining: 11.3s
100:	learn: 0.5741145	test: 0.5249278	best: 0.5249278 (100)	total: 85.3ms	remaining: 1.6s
200:	learn: 0.5665402	test: 0.5199401	best: 0.5199401 (200)	total: 158ms	remaining: 1.41s
300:	learn: 0.5605401	test: 0.5176867	best: 0.5176504 (293)	total: 233ms	remaining: 1.31s
400:	learn: 0.5561830	test: 0.5157032	best: 0.5156046 (393)	total: 308ms	remaining: 1.23s
500:	learn: 0.5533389	test: 0.5148584	best: 0.5147427 (489)	total: 383ms	remaining: 1.15s
600:	learn: 0.5509644	test: 0.5144305	best: 0.5143369 (594)	total: 458ms	remaining: 1.06s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5143369183
bestIteration = 594

Shrink model to first 595 iterations.


CatBoostClassifier(allow_writing_files=False, bagging_temperature=1.0, depth=3, eval_metric='CrossEntropy', iterations=2000, l2_leaf_reg=10, learning_rate=0.03, loss_function='CrossEntropy', random_seed=42, random_strength=1.0, verbose=100)

In [110]:
train_pred_prob_cb = cat_model.predict_proba(train_pool)[:, 1]
val_pred_prob_cb = cat_model.predict_proba(val_pool)[:, 1]

train_pred_score_cb = np.clip(train_pred_prob_cb * 100, 0, 100)
val_pred_score_cb = np.clip(val_pred_prob_cb * 100, 0, 100)

In [111]:
def regression_metrics(y_true_score, y_pred_score, name):
    mae = mean_absolute_error(y_true_score, y_pred_score)
    rmse = np.sqrt(mean_squared_error(y_true_score, y_pred_score))
    r2 = r2_score(y_true_score, y_pred_score)

    return {
        "split": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    }


regression_results_cb = pd.DataFrame([
    regression_metrics(train_pairs["ai_score"], train_pred_score_cb, "train"),
    regression_metrics(val_pairs["ai_score"], val_pred_score_cb, "validation"),
])

regression_results_cb

,split,MAE,RMSE,R2
0,train,4.773324,6.630811,0.912916
1,validation,9.647223,14.005839,0.708963


In [112]:
def campaign_ndcg(g):
    return ndcg_score(
        [g["ai_score"].values],
        [g["pred_ai_score"].values]
    )


def make_eval_df(pairs_df, pred_prob, pred_score):
    eval_df = pairs_df[[
        "campaign_original_index",
        "campaign_name",
        "blogger_folder_number",
        "blogger_username",
        "blogger_full_name",
        "ai_score",
        "score",
        "rank",
    ]].copy()

    eval_df["pred_prob"] = pred_prob
    eval_df["pred_ai_score"] = pred_score
    eval_df["abs_error"] = np.abs(eval_df["ai_score"] - eval_df["pred_ai_score"])

    return eval_df


train_eval_cb = make_eval_df(train_pairs, train_pred_prob_cb, train_pred_score_cb)
val_eval_cb = make_eval_df(val_pairs, val_pred_prob_cb, val_pred_score_cb)


def calc_ndcg_by_campaign(eval_df, name):
    ndcg_by_campaign = (
        eval_df
        .groupby("campaign_original_index")
        .apply(campaign_ndcg)
        .rename("ndcg")
        .reset_index()
    )

    mean_ndcg = ndcg_by_campaign["ndcg"].mean()

    print(f"{name} Mean NDCG: {mean_ndcg:.4f}")

    return ndcg_by_campaign, mean_ndcg


train_ndcg_by_campaign_cb, train_mean_ndcg_cb = calc_ndcg_by_campaign(train_eval_cb, "CatBoost Train")
val_ndcg_by_campaign_cb, val_mean_ndcg_cb = calc_ndcg_by_campaign(val_eval_cb, "CatBoost Validation")

CatBoost Train Mean NDCG: 0.9920
CatBoost Validation Mean NDCG: 0.9409


In [113]:
feature_importance_cb = pd.DataFrame({
    "feature": feature_cols,
    "importance": cat_model.get_feature_importance(train_pool)
}).sort_values("importance", ascending=False)

feature_importance_cb.head(30)

,feature,importance
15,retrieval_score,11.237368
14,emb_prod_sum,8.833018
9,emb_absdiff_mean,8.816245
2,emb_dot,7.033547
0,emb_cosine,6.854359
3,emb_euclidean,5.778856
1,emb_cosine_distance,5.582628
13,emb_prod_std,5.087241
12,emb_prod_mean,4.860368
10,emb_absdiff_std,4.412582


In [114]:
cat_model.save_model("catboost_soft_label_model.cbm")

train_eval_cb.to_excel("catboost_train_predictions.xlsx", index=False)
val_eval_cb.to_excel("catboost_validation_predictions.xlsx", index=False)

feature_importance_cb.to_excel("catboost_feature_importance.xlsx", index=False)

In [117]:
def build_account_features_df(df_profiles, profile_json_df):
    accounts = df_profiles.copy()

    # Нормализуем названия колонок
    accounts = accounts.rename(columns={
        "folder_number": "blogger_folder_number",
        "embedding": "profile_embedding",
        "username": "blogger_username",
        "full_name": "blogger_full_name",
        "followers": "blogger_followers",
        "blogger_gender": "blogger_gender",
        "domain": "blogger_domain",
        "keywords": "blogger_keywords",
        "entities": "blogger_entities",
        "brands": "blogger_brands",
        "video_format": "blogger_video_format",
        "visual_style": "blogger_visual_style",
        "audio_and_delivery": "blogger_audio_and_delivery",
        "vibe": "blogger_vibe",
        "rendered_text": "blogger_rendered_text",
    })

    accounts["blogger_folder_number"] = accounts["blogger_folder_number"].astype(int)

    # Мержим статистики из profile.json
    accounts = accounts.merge(
        profile_json_df,
        on="blogger_folder_number",
        how="left"
    )

    # Лог followers
    if "blogger_followers" in accounts.columns:
        accounts["blogger_followers"] = pd.to_numeric(
            accounts["blogger_followers"],
            errors="coerce"
        )
        accounts["blogger_followers_log"] = np.log1p(accounts["blogger_followers"])

    # Count-признаки по списочным текстовым полям
    text_list_cols = [
        "blogger_domain",
        "blogger_keywords",
        "blogger_entities",
        "blogger_visual_style",
        "blogger_brands",
        "blogger_video_format",
        "blogger_audio_and_delivery",
        "blogger_vibe",
    ]

    for col in text_list_cols:
        if col in accounts.columns:
            accounts[f"{col}_items_count"] = (
                accounts[col]
                .fillna("")
                .astype(str)
                .apply(lambda s: len([x for x in s.split(",") if x.strip()]))
            )

    # One-hot gender, можно потом использовать в CatBoost как числовые признаки
    if "blogger_gender" in accounts.columns:
        gender_dummies = pd.get_dummies(
            accounts["blogger_gender"].fillna("unknown"),
            prefix="gender",
            dtype=int
        )
        accounts = pd.concat([accounts, gender_dummies], axis=1)

    return accounts


account_features_df = build_account_features_df(df_profiles, profile_json_df)

account_features_df.shape

(284, 82)

In [118]:
assert "profile_embedding" in account_features_df.columns
assert account_features_df["profile_embedding"].notna().all()

account_features_df.head()

,blogger_folder_number,profile_dir,analysis_path,profile_id,blogger_username,blogger_full_name,blogger_followers,blogger_gender,blogger_domain,blogger_keywords,...,blogger_keywords_items_count,blogger_entities_items_count,blogger_visual_style_items_count,blogger_brands_items_count,blogger_video_format_items_count,blogger_audio_and_delivery_items_count,blogger_vibe_items_count,gender_женский,gender_мужской,gender_универсальный
0,1,1\5569396957,1\5569396957\gemini_analysis.json,5569396957,gvardis.m,Маша Гвардис | обзоры одежды и beauty | ugc | WB,3780,женский,"бьюти, косметика, мода, стиль, обзоры одежды, ...","блеск для губ, двухсторонний стик для лица, ск...",...,16,2,8,8,7,9,8,1,0,0
1,4,4\58159981827,4\58159981827\gemini_analysis.json,58159981827,as_crimeaa,Анастасия|Beauty|Обзоры|Севастополь,10228,женский,"бьюти-индустрия, уход за кожей, обзоры космети...","сыворотка для лица, сыворотка с витамином С, л...",...,18,4,11,11,7,11,10,1,0,0
2,5,5\53579029361,5\53579029361\gemini_analysis.json,53579029361,pon.alexa,Alexandra,13875,женский,"уход за волосами, бьюти-индустрия, красота, ух...","рост волос, фен для волос, термозащита для вол...",...,12,5,7,3,6,7,8,1,0,0
3,7,7\3685356465,7\3685356465\gemini_analysis.json,3685356465,kate_mom2,КАТЯ | мама двоих | UGC creator,1793,женский,"бьюти, уход за кожей, создание UGC-контента, м...","замороженный огурец для лица, натуральный масс...",...,20,1,8,5,6,7,8,1,0,0
4,9,9\65128479754,9\65128479754\gemini_analysis.json,65128479754,aammmmm79,Makeup model | FREE PALESTINA 🇵🇸,1622,женский,"бьюти-индустрия, профессиональный макияж, моде...","подводка для глаз, жидкий хайлайтер для скул, ...",...,11,5,7,1,4,6,8,1,0,0


In [119]:
def emb_dim(x):
    if isinstance(x, np.ndarray):
        return x.shape[0]
    if isinstance(x, list):
        return len(x)
    if isinstance(x, str):
        return len(np.fromstring(x.strip("[]"), sep=" "))
    return np.nan


account_features_df["profile_embedding_dim"] = account_features_df["profile_embedding"].apply(emb_dim)

account_features_df["profile_embedding_dim"].value_counts().head()

profile_embedding_dim
1024    284
Name: count, dtype: int64

In [120]:
ACCOUNT_FEATURES_PATH = "account_features_for_catboost.pkl"

account_features_df.to_pickle(ACCOUNT_FEATURES_PATH)

print(f"Saved: {ACCOUNT_FEATURES_PATH}")
print(account_features_df.shape)

Saved: account_features_for_catboost.pkl
(284, 83)


In [121]:
account_features_df = pd.read_pickle("account_features_for_catboost.pkl")

print(account_features_df.shape)
account_features_df.head()

(284, 83)


,blogger_folder_number,profile_dir,analysis_path,profile_id,blogger_username,blogger_full_name,blogger_followers,blogger_gender,blogger_domain,blogger_keywords,...,blogger_entities_items_count,blogger_visual_style_items_count,blogger_brands_items_count,blogger_video_format_items_count,blogger_audio_and_delivery_items_count,blogger_vibe_items_count,gender_женский,gender_мужской,gender_универсальный,profile_embedding_dim
0,1,1\5569396957,1\5569396957\gemini_analysis.json,5569396957,gvardis.m,Маша Гвардис | обзоры одежды и beauty | ugc | WB,3780,женский,"бьюти, косметика, мода, стиль, обзоры одежды, ...","блеск для губ, двухсторонний стик для лица, ск...",...,2,8,8,7,9,8,1,0,0,1024
1,4,4\58159981827,4\58159981827\gemini_analysis.json,58159981827,as_crimeaa,Анастасия|Beauty|Обзоры|Севастополь,10228,женский,"бьюти-индустрия, уход за кожей, обзоры космети...","сыворотка для лица, сыворотка с витамином С, л...",...,4,11,11,7,11,10,1,0,0,1024
2,5,5\53579029361,5\53579029361\gemini_analysis.json,53579029361,pon.alexa,Alexandra,13875,женский,"уход за волосами, бьюти-индустрия, красота, ух...","рост волос, фен для волос, термозащита для вол...",...,5,7,3,6,7,8,1,0,0,1024
3,7,7\3685356465,7\3685356465\gemini_analysis.json,3685356465,kate_mom2,КАТЯ | мама двоих | UGC creator,1793,женский,"бьюти, уход за кожей, создание UGC-контента, м...","замороженный огурец для лица, натуральный масс...",...,1,8,5,6,7,8,1,0,0,1024
4,9,9\65128479754,9\65128479754\gemini_analysis.json,65128479754,aammmmm79,Makeup model | FREE PALESTINA 🇵🇸,1622,женский,"бьюти-индустрия, профессиональный макияж, моде...","подводка для глаз, жидкий хайлайтер для скул, ...",...,5,7,1,4,6,8,1,0,0,1024


In [122]:
account_features_df.columns

Index(['blogger_folder_number', 'profile_dir', 'analysis_path', 'profile_id',
       'blogger_username', 'blogger_full_name', 'blogger_followers',
       'blogger_gender', 'blogger_domain', 'blogger_keywords',
       'blogger_entities', 'blogger_brands', 'blogger_video_format',
       'blogger_visual_style', 'blogger_audio_and_delivery', 'blogger_vibe',
       'blogger_rendered_text', 'profile_embedding', 'json_found',
       'json_followers', 'json_following', 'json_posts_count', 'json_verified',
       'json_private', 'json_business_account', 'json_videos_count',
       'json_following_to_followers', 'json_posts_to_followers',
       'json_days_since_last_post', 'json_posting_span_days',
       'json_posts_per_30d_in_sample', 'json_downloaded_share',
       'json_engagement_mean', 'json_engagement_median', 'json_likes_sum',
       'json_likes_mean', 'json_likes_median', 'json_likes_max',
       'json_likes_min', 'json_likes_std', 'json_comments_sum',
       'json_comments_mean', 'jso